In [1]:
import chromadb

# start chroma database
client = chromadb.PersistentClient("./chroma")

collection = client.create_collection("beauty_db")

docs = [
"niacinamide reduces oil and improves pores",
"azelaic acid treats acne and redness",
"retinol improves skin renewal",
"salicylic acid unclogs pores",
"hyaluronic acid hydrates skin",
"vitamin c brightens skin",
"benzoyl peroxide kills acne bacteria"
]

collection.add(
documents=docs,
ids=[f"doc{i}" for i in range(len(docs))]
)

print("Beauty database ready")

2026-03-08 16:04:21.831028441 [W:onnxruntime:Default, device_discovery.cc:131 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Beauty database ready


In [2]:
from agents import function_tool

@function_tool
def beauty_lookup(query: str) -> str:
    
    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    docs = results["documents"][0]

    if not docs:
        return "No skincare information found."

    return "\n".join(docs)

In [4]:
@function_tool
def find_beauty_store(product: str) -> str:
    
    stores = [
        "dm drogerie markt",
        "Müller",
        "Douglas"
    ]

    return f"You can buy {product} at: " + ", ".join(stores)

In [5]:
from agents import Agent

beauty_agent = Agent(
    name="Beauty Advisor",
    instructions="""
You are a helpful skincare advisor.

You help users with:
- skincare routines
- ingredient explanations
- where to buy products

Use the beauty_lookup tool when asked about ingredients.
Use find_beauty_store if the user asks where to buy something.

Keep answers short and clear.
""",
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[beauty_lookup, find_beauty_store]
)

In [6]:
from agents import Runner

result = await Runner.run(
    beauty_agent,
    "What does niacinamide do for skin?"
)

print(result.final_output)

Niacinamide is known to reduce oil production and improve the appearance of pores. It can also help in treating acne and reducing redness.


In [7]:
result = await Runner.run(
    beauty_agent,
    "Where can I buy niacinamide?"
)

print(result.final_output)

You can buy niacinamide at dm drogerie markt, Müller, or Douglas.


In [8]:
from agents import Agent, Runner, SQLiteSession

session = SQLiteSession("beauty_chat_history")

In [9]:
result = await Runner.run(
    beauty_agent,
    "I have oily skin and acne",
    session=session
)

print(result.final_output)

For oily skin and acne, consider using products with the following ingredients:

- **Benzoyl Peroxide**: Kills acne bacteria.
- **Vitamin C**: Brightens skin and reduces redness.
- **Azelaic Acid**: Treats acne and redness.

Would you like recommendations on specific products?


In [11]:
result = await Runner.run(
    beauty_agent,
    "What routine should I use?",
    session=session
)
print(result.final_output)

Here's a simple routine for oily skin and acne:

1. **Morning**:
   - **Cleanser**: Use a gentle cleanser with benzoyl peroxide or salicylic acid.
   - **Toner**: Apply a toner with vitamin C.
   - **Treatment**: Use a serum or spot treatment with azelaic acid.
   - **Moisturizer**: Apply a lightweight, non-comedogenic moisturizer.
   - **Sunscreen**: Use a broad-spectrum sunscreen with at least SPF 30.

2. **Evening**:
   - **Cleanser**: Use the same gentle cleanser with benzoyl peroxide or salicylic acid.
   - **Toner**: Apply the same toner with vitamin C.
   - **Treatment**: Use the same serum or spot treatment with azelaic acid.
   - **Moisturizer**: Apply the same lightweight, non-comedogenic moisturizer.

Would you like recommendations on specific products?
